# Random Forest Crop Recommendation Prototype

This notebook trains a six-feature prototype using N, P, K, pH, temperature, and rainfall.

Soil type and soil moisture are not used by this model because their source units and training signal are not approved. The Test split remains sealed. All reported results use Validation.


## 1 Setup


In [ ]:
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    ARTIFACT_DIR,
    FEATURES,
    RANDOM_STATE,
    TARGET,
    encode_training_validation_targets,
    evaluate_probabilities,
    load_training_validation_splits,
    plot_evaluation_dashboard,
    prediction_table,
    save_evaluation_outputs,
    training_validation_summary,
)

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
print("Python", platform.python_version())
print("Project root", PROJECT_ROOT)


## 2 Frozen Data Split


In [ ]:
train, validation, sealed_test_summary, split_integrity = load_training_validation_splits()
label_encoder, y_train, y_validation = encode_training_validation_targets(train, validation)

X_train = train[FEATURES].copy()
X_validation = validation[FEATURES].copy()

display(training_validation_summary(train, validation))
print("Features", FEATURES)
print("Crop classes", list(label_encoder.classes_))
print("Sealed Test metadata", sealed_test_summary)


In [ ]:
distribution = pd.concat(
    [
        train.assign(data_split="Train"),
        validation.assign(data_split="Validation"),
    ],
    ignore_index=True,
)
class_counts = (
    distribution.groupby([TARGET, "data_split"])
    .size()
    .rename("row_count")
    .reset_index()
)

plt.figure(figsize=(14, 7))
sns.barplot(data=class_counts, x=TARGET, y="row_count", hue="data_split")
plt.title("Frozen Split Class Distribution")
plt.xlabel("Crop Label")
plt.ylabel("Row Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 3 Train Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, top_k_accuracy_score

MODEL_NAME = "random_forest"
TOTAL_TREES = 400
TREE_STEP = 20

model = RandomForestClassifier(
    n_estimators=0,
    warm_start=True,
    max_features="sqrt",
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

history_rows = []
training_start = time.perf_counter()
progress = tqdm(
    range(TREE_STEP, TOTAL_TREES + 1, TREE_STEP),
    desc="Random Forest trees",
    unit="stage",
)
for tree_count in progress:
    model.set_params(n_estimators=tree_count)
    model.fit(X_train, y_train)
    stage_probabilities = model.predict_proba(X_validation)
    stage_loss = log_loss(
        y_validation, stage_probabilities, labels=np.arange(len(label_encoder.classes_))
    )
    stage_top_3 = top_k_accuracy_score(
        y_validation,
        stage_probabilities,
        k=3,
        labels=np.arange(len(label_encoder.classes_)),
    )
    elapsed = time.perf_counter() - training_start
    history_rows.append(
        {
            "iteration": tree_count,
            "training_loss": np.nan,
            "validation_loss": stage_loss,
            "validation_top_3": stage_top_3,
            "elapsed_seconds": elapsed,
        }
    )
    progress.set_postfix(top_3=f"{stage_top_3:.3f}", loss=f"{stage_loss:.3f}")

training_seconds = time.perf_counter() - training_start
training_history = pd.DataFrame(history_rows)
validation_probabilities = model.predict_proba(X_validation)
print("Random Forest training completed in", round(training_seconds, 3), "seconds")
display(training_history.tail())


## 4 Validation Evaluation


In [ ]:
validation_metrics, validation_per_class, validation_matrix, validation_calibration = (
    evaluate_probabilities(y_validation, validation_probabilities, label_encoder)
)
validation_predictions = prediction_table(
    y_validation, validation_probabilities, label_encoder
)

metrics_table = pd.DataFrame(
    {"metric": list(validation_metrics), "value": list(validation_metrics.values())}
)
display(metrics_table)
display(validation_per_class.sort_values("f1-score").reset_index(drop=True))

plot_evaluation_dashboard(
    MODEL_NAME,
    validation_metrics,
    validation_per_class,
    validation_matrix,
    validation_calibration,
    y_validation,
    validation_probabilities,
    label_encoder,
    training_history,
)


## 5 Feature Importance


In [ ]:
importance = pd.DataFrame(
    {"feature": FEATURES, "importance": model.feature_importances_}
).sort_values("importance", ascending=True)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance, x="importance", y="feature", color="#2f7d32")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 6 Save Validation Artifacts


In [ ]:
output_dir = save_evaluation_outputs(
    MODEL_NAME,
    validation_metrics,
    validation_per_class,
    validation_calibration,
    validation_predictions,
    training_history,
    training_seconds,
)
print("Saved evaluation artifacts to", output_dir)
print("Training time seconds", round(training_seconds, 3))
print("Test split used", False)


In [ ]:
joblib.dump(
    {
        "model": model,
        "label_encoder": label_encoder,
        "features": FEATURES,
        "data_contract_version": "prototype-six-feature-v1",
    },
    output_dir / "model.joblib",
)
print("Saved model", output_dir / "model.joblib")
